# 00 — Setup and Data Preparation

**Purpose.** This notebook bootstraps the Colab/local environment, clones the read-only
reference repositories, and prepares the datasets and normalization conventions shared by
every downstream notebook. It also defines the ImageNet-C corruption-robustness metric
(mCE) that later notebooks will report against.

**Math covered in this notebook.**
- Per-channel input normalization for the two model families used in this project.
- Train/val split conventions (CIFAR-10, ImageNet-1K/100).
- ImageNet-C per-corruption error $\mathrm{CE}_c$ and mean corruption error $\mathrm{mCE}$.

**Ablation IDs covered.** None — this is an infrastructure notebook (Aşama 0 in
`DENEY_PLANI_ve_ABLATIONS.md` Sec. 7).

**Inputs.** None (first notebook in the sequence).

**Outputs.**
- `external/` — read-only clones of VOneNet, CORnet, FOVEA.
- `data/` — CIFAR-10 (downloaded), plus ImageNet-1K/100 and the corruption/robustness
  benchmark sets if available locally (see Sec. "Dataset preparation" below for the
  honest fallback behavior when they are not).
- `results/00_environment.json` — environment stamp for provenance.
- `checkpoints/00_setup_complete.pt` — a marker checkpoint the next notebooks can check.

**Reference docs.** This notebook is the executable counterpart of `TEKNIK_REHBER.md`
Sec. 1–2 and `DENEY_PLANI_ve_ABLATIONS.md` Sec. 4. All planning documents are in Turkish;
this notebook (code, comments, and this text) is in English by project convention.


## Binding rules (recap)

1. **Colab is the target environment.** Every notebook starts with the same environment
   bootstrap cell (Drive mount if on Colab, local path otherwise).
2. **Notebooks are the unit of work.** Shared utilities live in `src/*.py` and are
   imported, but experiment logic and overrides stay visible in notebook cells.
3. **Never edit the cloned repositories.** Anything under `external/` is read-only
   reference code. Behavior changes are done via subclassing, monkey-patching, or
   wrapping — never by editing files inside `external/`.

See `TEKNIK_REHBER.md` for the full rationale.


In [ ]:
# --- Environment detection: Colab or local? ---
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/tez_foveated_vision"
else:
    PROJECT_ROOT = os.path.expanduser("~/Desktop/Bogazici/Tez/foveated_vision")

EXTERNAL_DIR = os.path.join(PROJECT_ROOT, "external")
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
CKPT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
for d in (PROJECT_ROOT, EXTERNAL_DIR, SRC_DIR, DATA_DIR, CKPT_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

import torch
print(f"Running in Colab: {IN_COLAB}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# Pinned dependencies (avoids breakage from Colab image drift). Safe to re-run.
!pip -q install torch==2.* torchvision timm==1.* foolbox==3.* robustbench \
    scipy scikit-learn matplotlib tqdm requests


In [ ]:
# Make `src/` importable as a package (`import src.common as common`, etc.)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.common as common
import src.datasets as datasets
import src.eval_harness as eval_harness

print("Imported src.common, src.datasets, src.eval_harness successfully.")


## Reference repository cloning (override policy)

The repositories below are cloned **read-only** into `external/` and added to
`sys.path`. We never edit files inside `external/`. Any behavioral change needed for an
ablation (e.g. disabling VOneBlock's Poisson noise, or swapping the ventral back-end)
is implemented as a subclass, a runtime monkey-patch, or a wrapper `nn.Module`, defined
in `src/overrides.py` (added in `01_baseline_reproduce.ipynb`, where it is first used).
This keeps every override a single reversible flag instead of a source edit.

Only the repos needed by the earliest notebooks are cloned here; the rest
(`LCL-V2`, `model_metamers_pytorch`, `GFNet-Pytorch`, `NeVA`, `NeuroFovea_PyTorch`,
`neural_manifolds_replicaMFT`, `adversarial-manifolds`) are cloned in the notebook that
first uses them, per `MAKALE_REPOLARI.md` Sec. 3.


In [ ]:
REPOS = {
    "vonenet": "https://github.com/dicarlolab/vonenet.git",
    "CORnet": "https://github.com/dicarlolab/CORnet.git",
    "fovea": "https://github.com/tchittesh/fovea.git",
}

for name, url in REPOS.items():
    dst = os.path.join(EXTERNAL_DIR, name)
    if not os.path.exists(dst):
        print(f"Cloning {name} ...")
        subprocess.run(["git", "clone", "--depth", "1", url, dst], check=True)
    else:
        print(f"{name} already present at {dst}")
    if dst not in sys.path:
        sys.path.insert(0, dst)


## Configuration and seeding

`CFG['smoke_test'] = True` keeps every dataset small (or synthetic, if the real one
isn't present) so this notebook — and every notebook built on top of it — can run
start to finish in a few minutes for validation. Flip it to `False` for a full run once
the real datasets are in place under `data/`.


In [ ]:
CFG = {
    "seed": 0,
    "smoke_test": True,       # True: tiny/synthetic data, fast end-to-end check
    "image_size": 224,        # matches VOneNet's default input_size / visual_degrees=8
    "data_dir": DATA_DIR,
}

common.set_seed(CFG["seed"])
print("CFG:", CFG)


## Math core: input normalization conventions

Every image is normalized per channel before entering a model:

$$
x'_c = \frac{x_c - \mu_c}{\sigma_c}, \qquad c \in \{R, G, B\}.
$$

Two different $(\mu, \sigma)$ conventions are used in this project, and picking the
wrong one for a given model silently degrades accuracy without raising an error:

- **VOneNet family** (VOneResNet-50, VOneAlexNet, VOneCORnet-S — baseline B2 and its
  ablations): $\mu = \sigma = (0.5, 0.5, 0.5)$, i.e. $x'_c = 2x_c - 1$, mapping
  $[0,1] \to [-1,1]$ (from the VOneNet reference implementation).
- **Plain torchvision backbones** (ResNet-50, AlexNet, CORnet-S baseline — baselines
  B0, B1, B3): standard ImageNet channel statistics,
  $\mu = (0.485, 0.456, 0.406)$, $\sigma = (0.229, 0.224, 0.225)$.

`src/datasets.py::build_transform(..., normalization="vonenet" | "imagenet")` implements
this equation directly and is used by every notebook that loads images. Forward
reference: VOneNet's default `image_size=224` with `visual_degrees=8` implies
$28$ pixels/degree — relevant once eccentricity enters degrees in
`02_foveation_rblur_and_periphery.ipynb`.


In [ ]:
import matplotlib.pyplot as plt

# Demonstrate build_transform on a dummy image and confirm the two conventions produce
# different normalized ranges, per the equation above.
dummy = torch.rand(3, 8, 8)
for norm in ("imagenet", "vonenet"):
    mean, std = datasets.normalization_stats(norm)
    normalized = (dummy - torch.tensor(mean).view(3, 1, 1)) / torch.tensor(std).view(3, 1, 1)
    print(f"{norm:9s} mean={mean} std={std}  ->  range=[{normalized.min():.3f}, {normalized.max():.3f}]")

## Math core: train / val split methodology

- **CIFAR-10**: fixed canonical split, 50,000 train / 10,000 test images
  (`torchvision.datasets.CIFAR10` manages this; no re-splitting).
- **ImageNet-1K / ImageNet-100**: the canonical ILSVRC train / val split is used as-is
  (val = 50,000 images, 50 per class) so results stay comparable to the literature —
  Brain-Score, ImageNet-C, and published baselines all assume this exact split. We do
  not carve a custom validation set out of train.
- **Fast-iteration subsets**: for ablation development, a small stratified subset of the
  train split ($k$ images/class) is used on ImageNet-100, mirroring the CIFAR-10-first
  workflow mandated by `DENEY_PLANI_ve_ABLATIONS.md` Sec. 4 ("Metodolojik kural").


## Math core: ImageNet-C corruption error and mCE

ImageNet-C (Hendrycks & Dietterich, 2019) applies $15$ corruption types
$c \in \mathcal{C}$ — grouped into noise (gaussian, shot, impulse), blur (defocus,
glass, motion, zoom), weather (snow, frost, fog, brightness), and digital (contrast,
elastic, pixelate, jpeg) — each at $5$ severities $s \in \{1, \dots, 5\}$.

Let $E_{s,c}$ be the evaluated model's top-1 **error rate** at corruption $c$, severity
$s$, and $E_{s,c}^{\text{AlexNet}}$ the same for the fixed AlexNet reference model. The
**per-corruption Corruption Error** is

$$
\mathrm{CE}_c \;=\; \frac{\displaystyle\sum_{s=1}^{5} E_{s,c}}{\displaystyle\sum_{s=1}^{5} E_{s,c}^{\text{AlexNet}}},
$$

normalized so that $\mathrm{CE}_c = 1$ means "as robust to corruption $c$ as AlexNet"
and $\mathrm{CE}_c < 1$ means more robust. The **mean Corruption Error** is

$$
\mathrm{mCE} \;=\; \frac{1}{15}\sum_{c=1}^{15} \mathrm{CE}_c.
$$

`src/eval_harness.py::corruption_error` and `::mean_corruption_error` implement these
two equations directly. No trained model exists yet at this stage of the pipeline, so
the cell below validates the implementation against small **synthetic, hand-picked**
error tables — not a real evaluation. Real $E_{s,c}$ tables are produced starting in
`01_baseline_reproduce.ipynb`.


In [ ]:
import random as _random

# Synthetic (toy) error tables purely to check the formula's arithmetic -- NOT a real
# evaluation of any model. Real corruption error tables are computed in later notebooks
# once trained baselines exist.
_rng = _random.Random(0)
toy_model_errors = {
    c: {s: 0.30 + 0.05 * s + 0.01 * _rng.random() for s in eval_harness.SEVERITY_LEVELS}
    for c in eval_harness.CORRUPTION_TYPES
}
toy_baseline_errors = {
    c: {s: 0.40 + 0.06 * s for s in eval_harness.SEVERITY_LEVELS}
    for c in eval_harness.CORRUPTION_TYPES
}

ce = eval_harness.corruption_error(toy_model_errors, toy_baseline_errors)
mce = eval_harness.mean_corruption_error(ce)

print("Per-corruption CE_c (toy numbers, sanity check only):")
for c in eval_harness.CORRUPTION_TYPES[:3]:
    print(f"  {c:20s} CE_c = {ce[c]:.3f}")
print("  ...")
print(f"mCE (toy) = {mce:.3f}  (expected < 1 since toy_model_errors < toy_baseline_errors)")
assert mce < 1.0, "Sanity check failed: toy model should score below the toy baseline."


## Dataset preparation

| Dataset | Role | Stage |
|---|---|---|
| CIFAR-10 | fast iteration / prototyping | 2–4 |
| ImageNet-1K | main clean accuracy and scale | 1–5 |
| ImageNet-C | common corruption robustness (mCE) | 1–5 |
| ImageNet-R / Sketch | domain shift, OOD | 3–5 |
| Stylized-ImageNet (Geirhos) | shape bias / cue-conflict | 5 |
| ImageNet-100 | FoveaTer-style efficiency comparison | 2, 4 |

(See `DENEY_PLANI_ve_ABLATIONS.md` Sec. 4 for the full table.) CIFAR-10 downloads
automatically below. The ImageNet-scale sets are large, license-gated, or hosted across
several academic mirrors — this notebook prepares them **honestly**: it uses them if
already present under `data/`, and otherwise prints the exact expected layout and skips
gracefully (falling back to a clearly-labeled synthetic stand-in when
`CFG['smoke_test']` is True) rather than failing or fabricating data.


In [ ]:
# In smoke_test mode this uses an already-cached copy if present, otherwise falls back
# to a synthetic stand-in rather than blocking on the (occasionally slow) official
# mirror -- same honest-fallback policy as ImageNet below. Set CFG['smoke_test'] = False
# for a real run, which always performs the real ~170 MB download.
cifar_train = datasets.get_cifar10(DATA_DIR, train=True, normalization="imagenet",
                                    download=True, smoke_test=CFG["smoke_test"])
cifar_test = datasets.get_cifar10(DATA_DIR, train=False, normalization="imagenet",
                                   download=True, smoke_test=CFG["smoke_test"])
print(f"CIFAR-10: {len(cifar_train)} train / {len(cifar_test)} test images "
      f"({type(cifar_train).__name__}).")


### ImageNet-1K and ImageNet-100

Not auto-downloaded (license-gated, ~150 GB for the full set). `get_imagenet` /
`get_imagenet100` look for a canonical `ImageFolder` layout under `data/imagenet` and
`data/imagenet100`; if missing, they fall back to a synthetic pipeline check when
`smoke_test=True`, or raise a clear, actionable error otherwise.


In [ ]:
imagenet_val = datasets.get_imagenet(
    DATA_DIR, split="val", normalization="imagenet",
    image_size=CFG["image_size"], smoke_test=CFG["smoke_test"],
)
imagenet100_val = datasets.get_imagenet100(
    DATA_DIR, split="val", normalization="imagenet",
    image_size=CFG["image_size"], smoke_test=CFG["smoke_test"],
)
print(f"ImageNet-1K val: {len(imagenet_val)} samples ({type(imagenet_val).__name__})")
print(f"ImageNet-100 val: {len(imagenet100_val)} samples ({type(imagenet100_val).__name__})")


### ImageNet-C / ImageNet-R / ImageNet-Sketch / Stylized-ImageNet

These are stable, well-known academic artifacts, but their exact download URLs move
between Zenodo/Drive/Kaggle hosts over time and this notebook cannot verify a URL is
still live without actually fetching it. `prepare_benchmark_dataset` downloads and
extracts whatever URL(s) you supply and otherwise prints the expected local layout and
skips — it never fabricates data or fails silently.

Two of the four sets are filled in below with URLs from their well-established,
long-standing hosts (Hendrycks' Berkeley page for ImageNet-R; the Zenodo record for
ImageNet-C's four category tars: blur/digital/noise/weather) — **verify these still
resolve before a full, non-smoke-test run**, since hosting can change. The other two
(ImageNet-Sketch, Stylized-ImageNet) are distributed via a Google Drive folder and a
style-transfer generation script respectively, not a plain static archive URL, so no
direct link is guessed here — see the printed instructions if you need them.


In [ ]:
BENCHMARK_URLS = {
    # Category tars hosted on Zenodo (Hendrycks & Dietterich, 2019). VERIFY before a
    # full run -- Zenodo occasionally renumbers/redirects records.
    "imagenet-c": [
        "https://zenodo.org/record/2235448/files/blur.tar",
        "https://zenodo.org/record/2235448/files/digital.tar",
        "https://zenodo.org/record/2235448/files/noise.tar",
        "https://zenodo.org/record/2235448/files/weather.tar",
    ],
    # Hendrycks et al., hosted on the author's Berkeley page. VERIFY before a full run.
    "imagenet-r": "https://people.eecs.berkeley.edu/~hendrycks/imagenet-r.tar",
    # No confident static archive URL: distributed via Google Drive by the authors.
    # See https://github.com/HaohanWang/ImageNet-Sketch for the current download link.
    "imagenet-sketch": None,
    # No confident static archive URL: typically generated locally via the authors'
    # style-transfer script rather than distributed as a prebuilt tarball (ImageNet
    # redistribution licensing). See https://github.com/rgeirhos/Stylized-ImageNet.
    "stylized-imagenet": None,
}

benchmark_status = {}
for name, urls in BENCHMARK_URLS.items():
    if CFG["smoke_test"] and urls is None:
        print(f"{name}: skipped in smoke_test mode (no URL available -- see comment above).")
        benchmark_status[name] = False
        continue
    if CFG["smoke_test"]:
        print(f"{name}: skipped in smoke_test mode (would download a multi-GB archive). "
              "Set CFG['smoke_test'] = False for a full run.")
        benchmark_status[name] = False
        continue
    benchmark_status[name] = datasets.prepare_benchmark_dataset(name, DATA_DIR, urls=urls)

print("\nBenchmark dataset availability:", benchmark_status)


In [ ]:
# Confirm the current src/ skeleton imports cleanly. overrides.py, foveation.py,
# it_feedback.py and mftma.py are added alongside 01/02/04/05 respectively, where their
# logic is first needed -- see src/__init__.py for the full planned layout.
import importlib

for module_name in ("src.common", "src.datasets", "src.eval_harness"):
    mod = importlib.import_module(module_name)
    print(f"OK: {module_name} ({mod.__file__})")


## Environment stamp and setup checkpoint

Per the reproducibility rules, every notebook records the exact environment that
produced its numbers. This notebook also writes a small "setup complete" checkpoint
that later notebooks can check for, and that captures the CFG used to prepare the data
(e.g. whether it ran in `smoke_test` mode).


In [ ]:
env_stamp = common.environment_stamp(PROJECT_ROOT)
env_stamp["cfg"] = CFG
env_stamp["benchmark_datasets_available"] = benchmark_status
common.save_json(env_stamp, os.path.join(RESULTS_DIR, "00_environment.json"))

common.save_checkpoint(
    {"setup_complete": True, "cfg": CFG, "timestamp_utc": env_stamp["timestamp_utc"]},
    os.path.join(CKPT_DIR, "00_setup_complete.pt"),
)

print("Environment stamp:")
for k, v in env_stamp.items():
    print(f"  {k}: {v}")


## Summary and handoff

Set up so far:
- `external/vonenet`, `external/CORnet`, `external/fovea` — cloned, read-only, on `sys.path`.
- `data/cifar10` — ready (canonical 50k/10k split).
- `data/imagenet(100)` — ready if present locally, else a documented synthetic fallback
  under `smoke_test=True`.
- `src/common.py`, `src/datasets.py`, `src/eval_harness.py` — importable, tested above.
- `results/00_environment.json`, `checkpoints/00_setup_complete.pt` — provenance markers.

**Next:** `01_baseline_reproduce.ipynb` — reproduces baselines B0–B3 (ResNet-50,
AlexNet, VOneResNet-50, CORnet-S), introduces `src/overrides.py` and the VOneBlock LNP
math (Gabor filters, quadrature complex cells, neuronal stochasticity), and validates
one robustness slice against the literature.
